# Más allá de la tabla · Notebook 1: carga y limpieza de datos

**Proyecto integrador de primer semestre.** Lógica Computacional e Introducción a la Ingeniería de Datos e IA.

**Objetivo de este notebook:** cargar los resultados de cinco temporadas de la Premier League, revisar su calidad, aplicar las decisiones de limpieza aprobadas por el equipo y guardar un único archivo limpio para las fases siguientes.

**Fuente de los datos:** English Premier League, publicado en DataHub a partir de Football-Data.co.uk, con licencia de dominio público (ODC-PDDL-1.0).
https://datahub.io/football/english-premier-league

**Decisiones aprobadas por el equipo**

| # | Decisión |
|---|---|
| 1 | Se analizan cinco temporadas: 2020/21, 2021/22, 2022/23, 2023/24 y 2024/25. |
| 2 | Se excluye la columna del árbitro (`Referee`): no se usa en la pregunta guía y sus nombres no son consistentes entre temporadas. |
| 3 | Si en un partido un equipo tiene más tiros al arco que tiros (dato imposible), se vacían **solo** esos dos valores de ese equipo. El resto del partido no cambia y el marcador sigue contando. |

**Cómo usarlo:** ejecuten las celdas en orden, de arriba hacia abajo (`Shift + Enter`). Cada paso tiene una explicación antes del código.

## Paso 0. Subir los archivos y preparar el entorno

1. Abran el panel de archivos de Colab (icono de carpeta en la barra de la izquierda).
2. Arrastren los cinco CSV descargados de DataHub, sin cambiarles el nombre.
3. Ejecuten la celda siguiente. Revisa que estén los cinco archivos y se detiene con un mensaje claro si falta alguno.

Los archivos subidos así se borran cuando se cierra la sesión de Colab: hay que volver a subirlos cada vez.

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Carpeta donde están los CSV. En Colab, los archivos subidos quedan en la carpeta actual.
CARPETA_DATOS = Path(".")

# Diccionario: temporada -> nombre del archivo descargado de DataHub.
TEMPORADAS = {
    "2020/21": "season-2021.csv",
    "2021/22": "season-2122.csv",
    "2022/23": "season-2223.csv",
    "2023/24": "season-2324.csv",
    "2024/25": "season-2425.csv",
}

archivos_faltantes = []
for temporada, archivo in TEMPORADAS.items():
    ruta = CARPETA_DATOS / archivo
    if ruta.exists():
        print(f"OK     {temporada}: {archivo}")
    else:
        print(f"FALTA  {temporada}: {archivo}")
        archivos_faltantes.append(archivo)

if archivos_faltantes:
    raise FileNotFoundError(f"Suban estos archivos antes de continuar: {archivos_faltantes}")

**Qué hace y por qué:**

- `TEMPORADAS` es un **diccionario**: a cada temporada le corresponde su archivo. Así el nombre de los archivos se escribe una sola vez y el resto del notebook lo reutiliza.
- El **bucle `for`** recorre las cinco temporadas y el **condicional `if`** separa los archivos encontrados de los que faltan.
- Si falta algún archivo, el notebook se detiene con un error que dice cuál subir. Es mejor parar aquí que obtener resultados incompletos sin darse cuenta.

## Paso 1. Conocer una temporada

Antes de trabajar con las cinco temporadas, miramos una sola para entender cómo vienen los datos. Usamos las tres funciones básicas de pandas para la exploración inicial.

In [ ]:
primera = pd.read_csv(CARPETA_DATOS / TEMPORADAS["2020/21"])
primera.head()

`head()` muestra las primeras cinco filas. Cada fila es **un partido**, con el equipo local (`HomeTeam`), el visitante (`AwayTeam`), el marcador y las estadísticas de cada equipo.

In [ ]:
primera.info()

`info()` muestra cuántas filas y columnas hay, cuántos valores no vacíos tiene cada columna y de qué tipo es cada una:

- **Números enteros** (`int64`): goles, tiros, faltas, córners y tarjetas.
- **Texto** (`object` o `str`, según la versión de pandas): fecha, equipos, resultados y árbitro.

Fíjense en que la fecha (`Date`) se lee como texto. En el paso 2 la convertimos a tipo fecha para poder ordenarla y compararla.

In [ ]:
primera.describe()

`describe()` calcula, para cada columna numérica, la cantidad de valores (`count`), la media (`mean`), la desviación estándar (`std`), el mínimo, los cuartiles y el máximo.

En esta fase **no** lo usamos para sacar conclusiones, sino para detectar valores raros: negativos, máximos imposibles o columnas con menos valores que filas. El análisis estadístico va en el notebook siguiente.

**Diccionario de variables**

| Columna | Significado |
|---|---|
| `Date` | Fecha del partido |
| `HomeTeam`, `AwayTeam` | Equipo local y equipo visitante |
| `FTHG`, `FTAG` | Goles al final del partido: local y visitante |
| `FTR` | Resultado final: `H` gana el local, `D` empate, `A` gana el visitante |
| `HTHG`, `HTAG`, `HTR` | Goles y resultado al medio tiempo |
| `Referee` | Árbitro (se excluye en el paso 4) |
| `HS`, `AS` | Tiros: local y visitante |
| `HST`, `AST` | Tiros al arco: local y visitante |
| `HF`, `AF` | Faltas cometidas: local y visitante |
| `HC`, `AC` | Córners: local y visitante |
| `HY`, `AY` | Tarjetas amarillas: local y visitante |
| `HR`, `AR` | Tarjetas rojas: local y visitante |

## Paso 2. Cargar las cinco temporadas

Escribimos una **función** que carga un archivo y le hace dos cosas:

1. Convierte la fecha de texto a tipo fecha.
2. Agrega una columna `Temporada`. Es importante hacerlo **antes** de unir los archivos; si no, después no sabríamos de qué temporada viene cada partido.

Guardamos cada temporada por separado en un diccionario. Las unimos solo al final, después de validarlas y limpiarlas.

In [ ]:
def cargar_temporada(ruta_csv, temporada):
    """Lee el CSV de una temporada y le agrega la columna "Temporada".

    Args:
        ruta_csv: ruta del archivo CSV.
        temporada: texto de la temporada, por ejemplo "2020/21".

    Returns:
        DataFrame con los partidos de esa temporada.
    """
    df = pd.read_csv(ruta_csv)
    # errors="coerce": si una fecha no se puede leer, queda vacía (NaT)
    # y la detectamos en la validación. No se reemplaza por otra fecha.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="coerce")
    df.insert(0, "Temporada", temporada)
    return df


datos = {}
for temporada, archivo in TEMPORADAS.items():
    datos[temporada] = cargar_temporada(CARPETA_DATOS / archivo, temporada)
    filas, columnas = datos[temporada].shape
    print(f"{temporada}: {filas} filas, {columnas} columnas")

Cada temporada debe tener **380 filas** (20 equipos que juegan todos contra todos, una vez de local y otra de visitante: 20 × 19 = 380) y **23 columnas** (las 22 del archivo más `Temporada`).

## Paso 3. Validar cada temporada

Revisamos la calidad de los datos con **funciones pequeñas, una por comprobación**. Ninguna modifica los datos: solo cuentan o buscan problemas. Así cada comprobación se puede explicar y probar por separado.

In [ ]:
COLUMNAS_ESPERADAS = [
    "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
    "HTHG", "HTAG", "HTR", "Referee", "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR",
]

COLUMNAS_NUMERICAS = [
    "FTHG", "FTAG", "HTHG", "HTAG", "HS", "AS", "HST", "AST",
    "HF", "AF", "HC", "AC", "HY", "AY", "HR", "AR",
]


def columnas_coinciden(df):
    """Indica si el archivo tiene exactamente las columnas esperadas, en el mismo orden."""
    columnas = list(df.columns.drop("Temporada"))
    return columnas == COLUMNAS_ESPERADAS


def columnas_no_numericas(df):
    """Devuelve las columnas que deberían ser números y no lo son."""
    problemas = []
    for columna in COLUMNAS_NUMERICAS:
        if not pd.api.types.is_numeric_dtype(df[columna]):
            problemas.append(columna)
    return problemas


def contar_nulos(df):
    """Cuenta las celdas vacías de toda la tabla."""
    return int(df.isna().sum().sum())


def contar_cruces_repetidos(df):
    """Cuenta los partidos repetidos.

    En una temporada cada equipo recibe a cada rival una sola vez,
    así que la pareja (local, visitante) no se puede repetir.
    """
    return int(df.duplicated(["HomeTeam", "AwayTeam"]).sum())


def contar_fechas_fuera_de_temporada(df, temporada):
    """Cuenta las fechas vacías o que no caen entre agosto y junio de la temporada."""
    anio_inicio = int(temporada[:4])
    inicio = pd.Timestamp(f"{anio_inicio}-08-01")
    fin = pd.Timestamp(f"{anio_inicio + 1}-06-30")
    fuera = df["Date"].isna() | (df["Date"] < inicio) | (df["Date"] > fin)
    return int(fuera.sum())


def equipos_con_calendario_incompleto(df):
    """Devuelve los equipos que no jugaron 19 partidos de local y 19 de visitante."""
    partidos_local = df["HomeTeam"].value_counts()
    partidos_visitante = df["AwayTeam"].value_counts()
    equipos = sorted(set(df["HomeTeam"]) | set(df["AwayTeam"]))
    incompletos = []
    for equipo in equipos:
        local = int(partidos_local.get(equipo, 0))
        visitante = int(partidos_visitante.get(equipo, 0))
        if local != 19 or visitante != 19:
            incompletos.append(f"{equipo}: {local} local, {visitante} visitante")
    return incompletos


def resultado_esperado(goles_local, goles_visitante):
    """Devuelve el resultado que corresponde a un marcador: "H", "D" o "A"."""
    if goles_local > goles_visitante:
        return "H"
    elif goles_local < goles_visitante:
        return "A"
    else:
        return "D"


def contar_resultados_incoherentes(df):
    """Cuenta los partidos cuyo resultado (FTR) no coincide con los goles."""
    incoherentes = 0
    for goles_local, goles_visitante, resultado in zip(df["FTHG"], df["FTAG"], df["FTR"]):
        if resultado != resultado_esperado(goles_local, goles_visitante):
            incoherentes += 1
    return incoherentes


def buscar_tiros_imposibles(df):
    """Busca equipos con más tiros al arco que tiros en un partido.

    Returns:
        Lista de diccionarios, uno por cada caso encontrado.
    """
    casos = []
    for indice in df.index:
        fila = df.loc[indice]
        partido = f"{fila['HomeTeam']} {fila['FTHG']}-{fila['FTAG']} {fila['AwayTeam']}"
        if fila["HST"] > fila["HS"]:
            casos.append({"indice": int(indice), "fecha": str(fila["Date"].date()),
                          "partido": partido, "equipo": fila["HomeTeam"],
                          "condicion": "local", "tiros": int(fila["HS"]),
                          "tiros_al_arco": int(fila["HST"])})
        if fila["AST"] > fila["AS"]:
            casos.append({"indice": int(indice), "fecha": str(fila["Date"].date()),
                          "partido": partido, "equipo": fila["AwayTeam"],
                          "condicion": "visitante", "tiros": int(fila["AS"]),
                          "tiros_al_arco": int(fila["AST"])})
    return casos


def contar_goles_mayores_que_tiros_al_arco(df):
    """Cuenta los partidos con más goles que tiros al arco en algún equipo.

    No es un error: un autogol cuenta para el rival aunque no haya rematado.
    """
    casos = (df["FTHG"] > df["HST"]) | (df["FTAG"] > df["AST"])
    return int(casos.sum())

Ahora aplicamos todas las funciones a cada temporada con un bucle y reunimos los resultados en un diccionario `resumen`. Al final lo mostramos como una tabla, con una fila por temporada.

In [ ]:
def validar_temporada(df, temporada):
    """Aplica todas las comprobaciones a una temporada.

    Returns:
        Diccionario con el resultado de cada comprobación.
    """
    equipos = set(df["HomeTeam"]) | set(df["AwayTeam"])
    return {
        "filas": len(df),
        "equipos": len(equipos),
        "columnas_correctas": columnas_coinciden(df),
        "columnas_no_numericas": len(columnas_no_numericas(df)),
        "celdas_vacias": contar_nulos(df),
        "partidos_repetidos": contar_cruces_repetidos(df),
        "fechas_fuera": contar_fechas_fuera_de_temporada(df, temporada),
        "calendario_incompleto": len(equipos_con_calendario_incompleto(df)),
        "resultados_incoherentes": contar_resultados_incoherentes(df),
        "tiros_imposibles": len(buscar_tiros_imposibles(df)),
        "goles_mayores_que_tiros_al_arco": contar_goles_mayores_que_tiros_al_arco(df),
    }


resumen = {}
for temporada, df in datos.items():
    resumen[temporada] = validar_temporada(df, temporada)

pd.DataFrame(resumen).T

**Cómo leer la tabla.** Una temporada está bien si tiene 380 filas, 20 equipos, `columnas_correctas` en `True` y cero en todas las demás columnas, salvo las dos últimas:

- `tiros_imposibles`: casos en que un equipo tiene más tiros al arco que tiros. Se corrigen en el paso 4 según la decisión 3.
- `goles_mayores_que_tiros_al_arco`: es solo informativa. No es un error, porque un autogol cuenta para el rival aunque no haya rematado al arco. El dataset no dice qué goles fueron autogoles, así que lo declaramos como limitación.

**Resultado esperado** con los archivos de DataHub que auditamos: las cinco temporadas pasan todas las comprobaciones; hay **un** caso de tiros imposibles (en 2021/22) y **10** partidos con más goles que tiros al arco en total. Si ven algo distinto, es que los archivos cambiaron y hay que revisarlos antes de seguir.

La celda siguiente muestra el detalle de los tiros imposibles.

In [ ]:
for temporada, df in datos.items():
    for caso in buscar_tiros_imposibles(df):
        print(f"{temporada} | {caso['fecha']} | {caso['partido']} | "
              f"{caso['equipo']} ({caso['condicion']}): "
              f"{caso['tiros']} tiros y {caso['tiros_al_arco']} tiros al arco")

## Paso 4. Limpiar

Aplicamos **solo** las decisiones aprobadas por el equipo:

1. **Quitar la columna `Referee`.**
2. **Vaciar los tiros imposibles.** Si un equipo tiene más tiros al arco que tiros, se vacían sus dos valores (tiros y tiros al arco) en ese partido. No se inventa ningún número y el marcador sigue contando para la clasificación.

**Un detalle técnico que hay que poder explicar:** en pandas, una columna de números enteros no puede tener celdas vacías. Por eso las cuatro columnas de tiros (`HS`, `AS`, `HST`, `AST`) se convierten a números decimales antes de vaciar el dato. Un 12 pasa a verse como 12.0, pero el valor es el mismo.

Cada cambio queda registrado en una lista para guardarlo en el reporte del paso 5.

In [ ]:
COLUMNAS_TIROS = ["HS", "AS", "HST", "AST"]


def limpiar_temporada(df):
    """Aplica las decisiones de limpieza aprobadas a una temporada.

    Args:
        df: DataFrame de una temporada.

    Returns:
        Tupla (DataFrame limpio, lista de cambios realizados).
    """
    limpio = df.drop(columns=["Referee"])
    limpio[COLUMNAS_TIROS] = limpio[COLUMNAS_TIROS].astype(float)

    cambios = []
    for caso in buscar_tiros_imposibles(df):
        if caso["condicion"] == "local":
            columnas = ["HS", "HST"]
        else:
            columnas = ["AS", "AST"]
        limpio.loc[caso["indice"], columnas] = float("nan")
        caso["columnas_vaciadas"] = columnas
        cambios.append(caso)
    return limpio, cambios


datos_limpios = {}
todos_los_cambios = []
for temporada, df in datos.items():
    limpio, cambios = limpiar_temporada(df)
    datos_limpios[temporada] = limpio
    for cambio in cambios:
        cambio["temporada"] = temporada
        todos_los_cambios.append(cambio)

print(f"Cambios realizados: {len(todos_los_cambios)}")
for cambio in todos_los_cambios:
    print(f"- {cambio['temporada']} {cambio['partido']}: se vaciaron "
          f"{cambio['columnas_vaciadas']} de {cambio['equipo']} "
          f"(antes: {cambio['tiros']} tiros y {cambio['tiros_al_arco']} al arco)")

**Por qué buscamos los casos en los datos originales (`df`) y no en el limpio:** la función `buscar_tiros_imposibles` necesita los números originales para comparar. Además, así el registro guarda qué valores había antes del cambio, y cualquiera puede comprobar qué se modificó.

## Paso 5. Unir, verificar y guardar

Unimos las cinco temporadas en una sola tabla con `pd.concat` y comprobamos el resultado con `assert`. Un `assert` es una prueba automática: si la condición es falsa, el notebook se detiene con un error en lugar de seguir con datos incorrectos.

In [ ]:
partidos = pd.concat(list(datos_limpios.values()), ignore_index=True)
partidos = partidos.sort_values(["Temporada", "Date"], kind="stable").reset_index(drop=True)

print(f"Tabla final: {partidos.shape[0]} filas y {partidos.shape[1]} columnas")
print(partidos["Temporada"].value_counts().sort_index())

In [ ]:
# 5 temporadas x 380 partidos = 1.900 filas.
# 22 columnas originales - Referee + Temporada = 22 columnas.
assert partidos.shape == (1900, 22), "El tamaño de la tabla no es el esperado"
assert (partidos["Temporada"].value_counts() == 380).all(), "Alguna temporada no tiene 380 partidos"
assert "Referee" not in partidos.columns, "La columna Referee no se eliminó"

# Las únicas celdas vacías deben ser las que vaciamos a propósito en el paso 4.
vacias_esperadas = 2 * len(todos_los_cambios)
assert partidos.isna().sum().sum() == vacias_esperadas, "Hay celdas vacías inesperadas"

# Después de limpiar, ya no debe quedar ningún caso de tiros imposibles.
assert not ((partidos["HST"] > partidos["HS"]) | (partidos["AST"] > partidos["AS"])).any()

print("Todas las verificaciones pasaron.")

Guardamos dos archivos:

- `partidos_limpios.csv`: la tabla limpia, con un partido por fila. Es la entrada de los notebooks siguientes y de la app.
- `reporte_limpieza.json`: la fuente, las decisiones, el resultado de cada validación y los cambios realizados. Sirve como evidencia del proceso para el informe y la sustentación.

In [ ]:
partidos.to_csv("partidos_limpios.csv", index=False)

reporte = {
    "fuente": "https://datahub.io/football/english-premier-league (Football-Data.co.uk)",
    "temporadas": list(TEMPORADAS.keys()),
    "decisiones": [
        "Se excluye la columna Referee.",
        "Si un equipo tiene más tiros al arco que tiros en un partido, "
        "se vacían solo esos dos valores de ese equipo.",
    ],
    "validacion_por_temporada": resumen,
    "cambios": todos_los_cambios,
    "filas_finales": int(partidos.shape[0]),
    "columnas_finales": int(partidos.shape[1]),
}

with open("reporte_limpieza.json", "w", encoding="utf-8") as archivo:
    json.dump(reporte, archivo, ensure_ascii=False, indent=2)

print("Archivos guardados: partidos_limpios.csv y reporte_limpieza.json")

**Comprobación final:** volvemos a leer los dos archivos desde el disco, para confirmar que se guardaron bien y que cualquiera puede abrirlos.

In [ ]:
verificacion = pd.read_csv("partidos_limpios.csv")
with open("reporte_limpieza.json", encoding="utf-8") as archivo:
    reporte_leido = json.load(archivo)

print(f"CSV leído de nuevo: {verificacion.shape[0]} filas y {verificacion.shape[1]} columnas")
print(f"Cambios registrados en el JSON: {len(reporte_leido['cambios'])}")

## Para terminar

1. Descarguen `partidos_limpios.csv` y `reporte_limpieza.json` desde el panel de archivos (clic derecho → Descargar). Colab los borra al cerrar la sesión.
2. Estos dos archivos irán a la carpeta `data/processed/` del repositorio.

**Preguntas que deben poder responder en la sustentación**

- ¿Por qué se agrega la columna `Temporada` antes de unir los archivos?
- ¿Por qué las funciones de validación no modifican los datos?
- ¿Qué problema tenía el partido Newcastle–West Ham y por qué vaciamos el dato en lugar de corregirlo o borrar el partido?
- ¿Por qué los partidos con más goles que tiros al arco no se consideran errores?
- ¿Por qué las columnas de tiros pasan de números enteros a decimales?
- ¿Qué hace un `assert` y qué pasaría si uno fallara?